In [ ]:
import json
import os
import shutil

from typing import Dict, List, Tuple

import geopandas as gpd
import pandas as pd


def load_json(file_name):
    with open(file_name, 'r') as file:
        return json.load(file)


def load_and_merge_annotations(coco_labels_file: str, metadata_file: str, categories: Dict[int, str], group_prefix: str) -> Tuple[pd.DataFrame, gpd.GeoDataFrame]:
    coco_data = load_json(coco_labels_file)
    coco_data["annotations"] = [
        annot for annot in coco_data["annotations"] 
        if annot["category_id"] in categories.keys()
    ]

    images_df = pd.DataFrame(data=coco_data["images"]).set_index("id")[["file_name", "width", "height"]]
    images_df.insert(
        loc=1, 
        column="name", 
        value=[os.path.splitext(os.path.basename(file))[0] for file in images_df["file_name"]]
    )

    metadata_gdf = gpd.read_file(metadata_file)[["file_name", "group", "geometry"]]
    metadata_gdf["name"] = [os.path.splitext(file_name)[0] for file_name in metadata_gdf["file_name"]]
    metadata_gdf.set_index("name", inplace=True)
    metadata_gdf.drop(columns="file_name", inplace=True)
    metadata_gdf["group"] = group_prefix + metadata_gdf["group"].astype(str)

    annotations_df = pd.DataFrame(data=coco_data["annotations"]).set_index("id")

    annotations_df[["x_min", "y_min", "x_width", "y_height"]] = annotations_df["bbox"].apply(lambda row: pd.Series(data=row))
    annotations_df["area"] = annotations_df["x_width"] * annotations_df["y_height"]

    annotations_df = (
        annotations_df
        .join(images_df, on="image_id", how="left")
        .join(metadata_gdf, on="name", how="left")
    )

    return annotations_df, metadata_gdf


def count_categories(image_df: pd.DataFrame, categories: Dict[int, str]) -> pd.Series:
    counts = image_df["category_id"].value_counts()

    data = {
        categories[key]: counts.loc[key]
        if key in counts.index
        else 0
        for key in sorted(categories.keys())
    }

    return pd.Series(
        data
    )

In [ ]:
categories = {
    1: "zwerfafval",
    3: "afvalberg"
}

In [ ]:
label_file = "../datasets/experiments/zwerfafval/labels/labels_250514_300_v0.1.json"
metadata_file = "../datasets/experiments/zwerfafval/recording_2025-05-14_19-47-40/dataselectie_300.gpkg"

annotations_250514_df, metadata_250514_gdf = load_and_merge_annotations(label_file, metadata_file, categories, group_prefix="250514_")

In [ ]:
label_file = "../datasets/experiments/zwerfafval/labels/labels_260421_1000_v0.1.json"
metadata_file = "../datasets/experiments/zwerfafval/inwinning_260421_selectie_1000_v1.gpkg"

annotations_260421_df, metadata_260421_gdf = load_and_merge_annotations(label_file, metadata_file, categories, group_prefix="260421_")

In [ ]:
# annotations_df = pd.concat([annotations_250514_df, annotations_260421_df])
# metadata_gdf = pd.concat([metadata_250514_gdf, metadata_260421_gdf])
annotations_df = annotations_250514_df
metadata_gdf = metadata_250514_gdf

In [ ]:
annotations_df

In [ ]:
annotations_df[annotations_df["category_id"]==1].plot.scatter(x="x_min", y="y_min", ylim=[1, 0], c="area")

In [ ]:
annotations_df[annotations_df["category_id"]==1].plot.scatter(x="x_width", y="y_height")

In [ ]:
annotations_df.sort_values(by="area").tail(10)

In [ ]:
count_categories(annotations_df, categories=categories)

In [ ]:
dataset = annotations_df.groupby(by="name", group_keys=True).apply(func=count_categories, categories=categories)
dataset["group"] = (
    annotations_df
    .groupby(by="name", group_keys=True)
    .apply(lambda df: df["group"].iloc[0])
    .loc[dataset.index]
)

dataset

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter_train_val = GroupShuffleSplit(n_splits=1, test_size=0.4)
classes = list(categories.values())

X = dataset[["group"]]
y = dataset[classes]

train_ix, not_train_ix = next(splitter_train_val.split(X, y, groups=X["group"]))

In [ ]:
print(f"Train percentage: {len(train_ix) / len(dataset)}")

In [ ]:
train_distribution = dataset[classes].iloc[train_ix, :].sum()

print(train_distribution / dataset[classes].sum())

In [ ]:
not_train_data = dataset.iloc[not_train_ix, :]

splitter_test = GroupShuffleSplit(n_splits=1, test_size=0.5)

X = not_train_data[["group"]]
y = not_train_data[classes]

val_ix, test_ix = next(splitter_test.split(X, y, groups=X["group"]))

In [ ]:
print(f"Validation percentage: {len(val_ix) / len(dataset)}")
print(f"Test percentage: {len(test_ix) / len(dataset)}")

In [ ]:
val_distribution = not_train_data[classes].iloc[val_ix, :].sum()
test_distribution = not_train_data[classes].iloc[test_ix, :].sum()

print("Validation set:")
print(val_distribution / dataset[classes].sum())
print("\nTest set:")
print(test_distribution / dataset[classes].sum())

In [ ]:
dataset_train = dataset.iloc[train_ix, :]
dataset_val = not_train_data.iloc[val_ix, :]
dataset_test = not_train_data.iloc[test_ix, :]

In [ ]:
def coco_bbox_to_yolo(coco_bbox: List[float], img_width: int, img_height: int) -> List[float]:
    """
    Converts COCO bounding box format to YOLO format.

    Parameters
    ----------
    bbox : List[float]
        Bounding box in COCO format [x_min, y_min, width, height].
    img_width : int
        Width of the image.
    img_height : int
        Height of the image.

    Returns
    -------
    List[float]
        Bounding box in YOLO format [x_center, y_center, width, height], normalized.
    """
    x_min, y_min, width, height = coco_bbox
    try:
        x_center = x_min + width / 2
        y_center = y_min + height / 2

        return [x_center, y_center, width, height]
    except ZeroDivisionError:
        print(f"ZeroDivisionError encountered with: img_width={img_width}, img_height={img_height}, x_min={x_min}, y_min={y_min}, bbox_width={width}, bbox_height={height}")
        return None


def write_labels(dataset: pd.DataFrame, annotations_df: pd.DataFrame, output_folder: str):
    os.makedirs(output_folder, exist_ok=True)

    for name in dataset.index.values:
        yolo_string = ""
        for row in annotations_df[annotations_df["name"]==name].itertuples():
            yolo_bbox = coco_bbox_to_yolo(row.bbox, row.width, row.height)
            yolo_string += f"{row.category_id} {' '.join(map(str, yolo_bbox))}\n"
        
        file_path = os.path.join(output_folder, f"{name}.txt")
        with open(file_path, "w") as f:
            f.write(yolo_string)


images_folder = {
    "250514": "../datasets/experiments/zwerfafval/annotatieproject/inwinning_250514_selectie_300",
    "260421": "../datasets/experiments/zwerfafval/annotatieproject/inwinning_260421_selectie_1000",  # NOTE: typo is on purpose
}

def copy_images(dataset: pd.DataFrame, output_folder: str):
    os.makedirs(output_folder, exist_ok=True)

    for row in dataset.itertuples():
        prefix = row.group.split(sep="_")[0]
        source_file = os.path.join(images_folder[prefix], f"{row.Index}.jpg")
        shutil.copy2(source_file, output_folder)


def copy_background_images(dataset: pd.DataFrame, metadata_df: pd.DataFrame, output_folder: str):
    os.makedirs(output_folder, exist_ok=True)
    
    for group in dataset["group"].unique():
        all_names = set(metadata_df[metadata_df["group"]==group].index.values)
        background_names = all_names - set(dataset[dataset["group"]==group].index.values)
        
        for name in background_names:
            prefix = group.split(sep="_")[0]
            source_file = os.path.join(images_folder[prefix], f"{name}.jpg")
            shutil.copy2(source_file, output_folder)

In [ ]:
dataset_folder = "../datasets/experiments/zwerfafval/dataset_v0.1"

annotations_output_folder = os.path.join(dataset_folder, "labels")
images_output_folder = os.path.join(dataset_folder, "images")
background_output_folder = os.path.join(dataset_folder, "background_images")


dataset_parts: Dict[str, pd.DataFrame] = {
    "train": dataset_train,
    "val": dataset_val,
    "test": dataset_test,
}

for (part, part_dataset) in dataset_parts.items():
    write_labels(
        dataset=part_dataset,
        annotations_df=annotations_df,
        output_folder=os.path.join(annotations_output_folder, part)
    )

    copy_images(
        dataset=part_dataset,
        output_folder=os.path.join(images_output_folder, part)
    )

    copy_background_images(
        dataset=part_dataset,
        metadata_df=metadata_gdf,
        output_folder=os.path.join(background_output_folder, part)
    )

In [ ]:
# TODO

# manually select X background images per split
# add bg to splits
# generate yolo dataset yaml